# Đồ Án 1 – Phần 1: Phép Khử Gauss và Các Ứng Dụng
**Môn học:** Toán Ứng Dụng và Thống Kê (MTH00051)  
**Học kỳ:** HK2, 2025–2026  

---

Notebook này minh họa toàn bộ các hàm đã cài đặt trong Phần 1:
1. Phép khử Gauss có Partial Pivoting (`gaussian_eliminate`, `back_substitution`)
2. Tính định thức (`determinant`)
3. Ma trận nghịch đảo (`inverse`)
4. Hạng và cơ sở (`rank_and_basis`)
5. Kiểm chứng bằng NumPy (`verify_solution`)

## 0. Setup & Import

In [1]:
import sys, os
import numpy as np
from fractions import Fraction

# Import các module tự cài đặt
from gaussian import gaussian_eliminate, back_substitution
from determinant import determinant
from inverse import inverse
from rank_and_basic import rank_and_basis

print("Import thành công tất cả các module.")

Import thành công tất cả các module.


## 1. Hàm tiện ích

In [2]:
def print_matrix(M, title="Ma trận", precision=4):
    """In ma trận dạng đẹp."""
    print(f"\n{'='*40}")
    print(f"  {title}")
    print(f"{'='*40}")
    for row in M:
        formatted = []
        for val in row:
            if isinstance(val, Fraction):
                formatted.append(f"{float(val):>{precision+5}.{precision}f}")
            else:
                formatted.append(f"{float(val):>{precision+5}.{precision}f}")
        print("  [ " + "  ".join(formatted) + " ]")
    print()


def verify_solution(A, x_vec, b):
    """
    Kiểm chứng nghiệm x bằng NumPy.
    Tính sai số tương đối ||Ax - b|| / ||b||.
    """
    A_np = np.array(A, dtype=float)
    b_np = np.array(b, dtype=float)
    x_np = np.array(x_vec, dtype=float)

    residual = np.linalg.norm(A_np @ x_np - b_np)
    rel_error = residual / (np.linalg.norm(b_np) + 1e-15)

    x_ref = np.linalg.solve(A_np, b_np)
    sol_error = np.linalg.norm(x_np - x_ref)

    print(f"  Nghiệm tự cài đặt  : {np.round(x_np, 6).tolist()}")
    print(f"  Nghiệm NumPy       : {np.round(x_ref, 6).tolist()}")
    print(f"  Sai số tương đối   : {rel_error:.2e}")
    print(f"  ||x - x_ref||      : {sol_error:.2e}")
    status = "PASS" if rel_error < 1e-8 else " FAIL"
    print(f"  Kết quả            : {status}")
    return rel_error


def parse_unique_solution(result_str):
    """
    Trích xuất vector nghiệm từ chuỗi kết quả trả về bởi back_substitution.
    Ví dụ: 'Nghiệm duy nhất: x = [2.0, 3.0, -1.0]' -> [2.0, 3.0, -1.0]
    """
    import ast
    if "Nghiệm duy nhất" not in result_str:
        return None
    vec_str = result_str.split("x = ")[1].strip()
    return ast.literal_eval(vec_str)


print("Hàm tiện ích đã sẵn sàng.")

Hàm tiện ích đã sẵn sàng.


---
## 2. Phép Khử Gauss (`gaussian_eliminate` & `back_substitution`)

### 2.1 Lý thuyết tóm tắt

Cho hệ $Ax = b$ với $A \in \mathbb{R}^{m \times n}$. Phép khử Gauss:
1. Lập ma trận tăng cường $[A | b]$
2. Áp dụng partial pivoting: tại bước $k$, chọn hàng $p = \arg\max_{k \le i \le n} |M_{ik}|$
3. Khử về dạng bậc thang (REF)
4. Thế ngược để tìm nghiệm

Công thức định thức: $\det(A) = (-1)^s \cdot \prod_{i=1}^n u_{ii}$

### 2.2 Test cases — Giải hệ phương trình

In [3]:
print("" + "="*60)
print("  TEST 1: Hệ có nghiệm duy nhất (3x3)")
print("="*60)
# Hệ: 2x + y - z = 8 | -3x - y + 2z = -11 | -2x + y + 2z = -3
# Nghiệm đúng: x=2, y=3, z=-1
A1 = [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]]
b1 = [8, -11, -3]

print("\nMa trận A:")
print_matrix(A1, "A")
print(f"Vector b: {b1}")

M1, x1, s1 = gaussian_eliminate(A1, b1)
print(f"\nSố lần hoán đổi hàng (s): {s1}")
print(f"Kết quả: {x1}")

print("\n--- Kiểm chứng NumPy ---")
x1_vec = parse_unique_solution(x1)
verify_solution(A1, x1_vec, b1)

  TEST 1: Hệ có nghiệm duy nhất (3x3)

Ma trận A:

  A
  [    2.0000     1.0000    -1.0000 ]
  [   -3.0000    -1.0000     2.0000 ]
  [   -2.0000     1.0000     2.0000 ]

Vector b: [8, -11, -3]

Số lần hoán đổi hàng (s): 2
Kết quả: Nghiệm duy nhất: x = [2.0, 3.0, -1.0]

--- Kiểm chứng NumPy ---
  Nghiệm tự cài đặt  : [2.0, 3.0, -1.0]
  Nghiệm NumPy       : [2.0, 3.0, -1.0]
  Sai số tương đối   : 0.00e+00
  ||x - x_ref||      : 8.38e-16
  Kết quả            : PASS


np.float64(0.0)

In [4]:
print("" + "="*60)
print("  TEST 2: Hệ vô nghiệm")
print("="*60)
# Hàng 2 = 2 * hàng 1, nhưng b2 ≠ 2*b1
A2 = [[1, 2, 3], [2, 4, 6], [0, 1, 1]]
b2 = [1, 5, 2]
print(f"\nMa trận A: {A2}")
print(f"Vector b: {b2}")
M2, x2, s2 = gaussian_eliminate(A2, b2)
print(f"Kết quả: {x2}")
assert "vô nghiệm" in x2.lower() or "vô nghiệm" in str(x2), "Phải là hệ vô nghiệm!"
print("Phát hiện đúng: Hệ vô nghiệm.")

  TEST 2: Hệ vô nghiệm

Ma trận A: [[1, 2, 3], [2, 4, 6], [0, 1, 1]]
Vector b: [1, 5, 2]
Kết quả: Hệ vô nghiệm
Phát hiện đúng: Hệ vô nghiệm.


In [5]:
print("" + "="*60)
print("  TEST 3: Hệ vô số nghiệm (nghiệm tổng quát)")
print("="*60)
# 3 phương trình, 4 ẩn, hạng = 2 → 2 ẩn tự do
A3 = [[1, 2, 0, 1], [0, 0, 1, 3], [1, 2, 1, 4]]
b3 = [1, 2, 3]
print(f"\nMa trận A (3x4): {A3}")
print(f"Vector b: {b3}")
M3, x3, s3 = gaussian_eliminate(A3, b3)
print(f"Kết quả: {x3}")
assert "tổng quát" in x3.lower(), "Phải có vô số nghiệm!"
print("Phát hiện đúng: Hệ vô số nghiệm với nghiệm tổng quát.")

  TEST 3: Hệ vô số nghiệm (nghiệm tổng quát)

Ma trận A (3x4): [[1, 2, 0, 1], [0, 0, 1, 3], [1, 2, 1, 4]]
Vector b: [1, 2, 3]
Kết quả: Nghiệm tổng quát: x = [0.3333333333, 0.0, 0.0, 0.6666666667] + t1 * [-2.0, 1.0, -0.0, 0.0]
Phát hiện đúng: Hệ vô số nghiệm với nghiệm tổng quát.


In [6]:
print("" + "="*60)
print("  TEST 4: Ma trận 1x1")
print("="*60)
A4 = [[5]]
b4 = [15]
M4, x4, s4 = gaussian_eliminate(A4, b4)
print(f"Kết quả: {x4}")
x4_vec = parse_unique_solution(x4)
verify_solution(A4, x4_vec, b4)

  TEST 4: Ma trận 1x1
Kết quả: Nghiệm duy nhất: x = [3.0]
  Nghiệm tự cài đặt  : [3.0]
  Nghiệm NumPy       : [3.0]
  Sai số tương đối   : 0.00e+00
  ||x - x_ref||      : 0.00e+00
  Kết quả            : PASS


np.float64(0.0)

In [7]:
print("" + "="*60)
print("  TEST 5: Hệ 4x4 với partial pivoting")
print("="*60)
# Ma trận cần partial pivoting để tránh chia cho số nhỏ
A5 = [[0.0001, 1, 0, 0],
      [1,      2, 1, 0],
      [0,      1, 3, 1],
      [0,      0, 1, 4]]
b5 = [1, 4, 5, 5]
print(f"\nLưu ý: Hàng đầu có pivot rất nhỏ (0.0001) → cần partial pivoting.")
M5, x5, s5 = gaussian_eliminate(A5, b5)
print(f"Số lần hoán đổi: {s5}")
print(f"Kết quả: {x5}")
x5_vec = parse_unique_solution(x5)
verify_solution(A5, x5_vec, b5)

  TEST 5: Hệ 4x4 với partial pivoting

Lưu ý: Hàng đầu có pivot rất nhỏ (0.0001) → cần partial pivoting.
Số lần hoán đổi: 2
Kết quả: Nghiệm duy nhất: x = [1.0001636631, 0.9998999836, 1.0000363696, 0.9999909076]
  Nghiệm tự cài đặt  : [1.000164, 0.9999, 1.000036, 0.999991]
  Nghiệm NumPy       : [1.000164, 0.9999, 1.000036, 0.999991]
  Sai số tương đối   : 1.29e-11
  ||x - x_ref||      : 5.75e-11
  Kết quả            : PASS


np.float64(1.2891638603563944e-11)

---
## 3. Tính Định Thức (`determinant`)

Công thức: $\det(A) = (-1)^s \cdot \prod_{i=1}^n u_{ii}$  
với $s$ là số lần hoán đổi hàng và $u_{ii}$ là phần tử đường chéo sau khử Gauss.

In [8]:
def verify_determinant(A, label=""):
    det_custom = determinant(A)
    det_numpy  = np.linalg.det(np.array(A, dtype=float))
    error = abs(det_custom - det_numpy)
    status = "PASS" if error < 1e-8 else "FAIL"
    print(f"  {label}")
    print(f"    det (tự cài)  = {det_custom:.6f}")
    print(f"    det (NumPy)   = {det_numpy:.6f}")
    print(f"    Sai số tuyệt đối: {error:.2e}  {status}")
    return det_custom


print("="*60)
print("  TEST 1: Ma trận 2x2")
print("="*60)
# det([[1,2],[3,4]]) = 1*4 - 2*3 = -2
verify_determinant([[1, 2], [3, 4]], "[[1,2],[3,4]]")

print()
print("="*60)
print("  TEST 2: Ma trận 3x3")
print("="*60)
verify_determinant([[1, 2, 3], [0, 1, 4], [5, 6, 0]], "Ma trận 3x3")

print()
print("="*60)
print("  TEST 3: Ma trận kỳ dị (định thức = 0)")
print("="*60)
# Hàng 2 = 2 * hàng 1 → det = 0
verify_determinant([[1, 2, 3], [2, 4, 6], [0, 1, 4]], "Ma trận kỳ dị")

print()
print("="*60)
print("  TEST 4: Ma trận đơn vị 4x4 (det = 1)")
print("="*60)
I4 = [[1 if i==j else 0 for j in range(4)] for i in range(4)]
verify_determinant(I4, "Ma trận đơn vị 4x4")

print()
print("="*60)
print("  TEST 5: Ma trận 5x5 ngẫu nhiên")
print("="*60)
np.random.seed(42)
A_rand = np.random.randint(1, 10, (5, 5)).tolist()
verify_determinant(A_rand, "Ma trận 5x5 ngẫu nhiên (seed=42)")

  TEST 1: Ma trận 2x2
  [[1,2],[3,4]]
    det (tự cài)  = -2.000000
    det (NumPy)   = -2.000000
    Sai số tuyệt đối: 4.44e-16  PASS

  TEST 2: Ma trận 3x3
  Ma trận 3x3
    det (tự cài)  = 1.000000
    det (NumPy)   = 1.000000
    Sai số tuyệt đối: 0.00e+00  PASS

  TEST 3: Ma trận kỳ dị (định thức = 0)
  Ma trận kỳ dị
    det (tự cài)  = 0.000000
    det (NumPy)   = 0.000000
    Sai số tuyệt đối: 0.00e+00  PASS

  TEST 4: Ma trận đơn vị 4x4 (det = 1)
  Ma trận đơn vị 4x4
    det (tự cài)  = 1.000000
    det (NumPy)   = 1.000000
    Sai số tuyệt đối: 0.00e+00  PASS

  TEST 5: Ma trận 5x5 ngẫu nhiên


  Ma trận 5x5 ngẫu nhiên (seed=42)
    det (tự cài)  = -3060.000000
    det (NumPy)   = -3060.000000
    Sai số tuyệt đối: 3.18e-12  PASS


-3060.0

---
## 4. Ma Trận Nghịch Đảo (`inverse`)

Phương pháp Gauss–Jordan: biến đổi $[A | I_n] \to [I_n | A^{-1}]$  
Điều kiện: $\det(A) \neq 0$

In [9]:
def verify_inverse(A, label=""):
    """Kiểm chứng A^{-1} bằng cách kiểm tra A @ A^{-1} = I."""
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    try:
        A_inv = inverse(A)
        n = len(A)

        # Chuyển về float để tính toán
        A_inv_float = [[float(A_inv[i][j]) for j in range(n)] for i in range(n)]
        A_np  = np.array(A, dtype=float)
        Ainv_np = np.array(A_inv_float, dtype=float)

        # Kiểm tra A @ A^{-1} = I
        product = A_np @ Ainv_np
        identity = np.eye(n)
        error = np.linalg.norm(product - identity, 'fro')

        print("  A^{-1} (dạng phân số):")
        for row in A_inv:
            print("   ", [str(v) for v in row])

        print(f"\n  ||A @ A^{{-1}} - I||_F = {error:.2e}", "PASS" if error < 1e-8 else "FAIL")

        # So sánh với NumPy
        Ainv_ref = np.linalg.inv(A_np)
        diff = np.linalg.norm(Ainv_np - Ainv_ref, 'fro')
        print(f"  ||A^{{-1}} - A^{{-1}}_NumPy||_F = {diff:.2e}", "PASS" if diff < 1e-8 else "FAIL")

    except ValueError as e:
        print(f"  Lỗi (mong đợi): {e}")


# TEST 1: Ma trận 2x2 cơ bản
verify_inverse([[1, 2], [3, 4]], "TEST 1: Ma trận 2x2")

# TEST 2: Ma trận 3x3
verify_inverse([[1, 2, 5], [0, -2, -1], [0, -3, -7]], "TEST 2: Ma trận 3x3")

# TEST 3: Ma trận kỳ dị (không có nghịch đảo)
verify_inverse([[1, 2, 3], [2, 4, 6], [0, 1, 4]], "TEST 3: Ma trận kỳ dị (det=0) — Mong đợi lỗi")

# TEST 4: Ma trận đơn vị
verify_inverse([[1, 0, 0], [0, 1, 0], [0, 0, 1]], "TEST 4: Ma trận đơn vị 3x3")

# TEST 5: Ma trận không vuông
verify_inverse([[1, 2, 3], [4, 5, 6]], "TEST 5: Ma trận không vuông — Mong đợi lỗi")


  TEST 1: Ma trận 2x2
  A^{-1} (dạng phân số):
    ['-2', '1']
    ['3/2', '-1/2']

  ||A @ A^{-1} - I||_F = 0.00e+00 PASS
  ||A^{-1} - A^{-1}_NumPy||_F = 5.55e-16 PASS

  TEST 2: Ma trận 3x3
  A^{-1} (dạng phân số):
    ['1', '-1/11', '8/11']
    ['0', '-7/11', '1/11']
    ['0', '3/11', '-2/11']

  ||A @ A^{-1} - I||_F = 2.29e-16 PASS
  ||A^{-1} - A^{-1}_NumPy||_F = 1.87e-16 PASS

  TEST 3: Ma trận kỳ dị (det=0) — Mong đợi lỗi
  Lỗi (mong đợi): Ma trận có định thức = 0 nên không thể chéo hóa

  TEST 4: Ma trận đơn vị 3x3
  A^{-1} (dạng phân số):
    ['1', '0', '0']
    ['0', '1', '0']
    ['0', '0', '1']

  ||A @ A^{-1} - I||_F = 0.00e+00 PASS
  ||A^{-1} - A^{-1}_NumPy||_F = 0.00e+00 PASS

  TEST 5: Ma trận không vuông — Mong đợi lỗi
  Lỗi (mong đợi): Ma trận đầu vào buộc phải là ma trận vuông


---
## 5. Hạng và Cơ Sở (`rank_and_basis`)

Từ dạng RREF xác định:
- **Hạng** = số cột pivot
- **Không gian cột** $C(A)$: các cột pivot của ma trận **gốc** $A$
- **Không gian hàng** $R(A)$: các hàng khác 0 trong RREF
- **Không gian nghiệm** $N(A)$: nghiệm của $Ax = 0$ (ứng với các biến tự do)

In [10]:
def print_rank_result(A, label=""):
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(f"  Ma trận A ({len(A)}x{len(A[0])}):", A)

    rank, col_b, row_b, null_b = rank_and_basis(A)

    # So sánh với NumPy
    rank_np = np.linalg.matrix_rank(np.array(A, dtype=float))
    rank_status = "PASS" if rank == rank_np else "FAIL"
    print(f"\n  Hạng (tự cài): {rank}  |  Hạng (NumPy): {rank_np}  {rank_status}")

    print(f"\n  Cơ sở không gian cột C(A) ({len(col_b)} vector):")
    for v in col_b:
        print("   ", [str(x) for x in v])

    print(f"\n  Cơ sở không gian hàng R(A) ({len(row_b)} vector):")
    for v in row_b:
        print("   ", [str(x) for x in v])

    print(f"\n  Cơ sở không gian nghiệm N(A) ({len(null_b)} vector):")
    if null_b:
        for v in null_b:
            print("   ", [str(x) for x in v])
        # Kiểm chứng: Av = 0 với mọi v trong null_b
        A_np = np.array(A, dtype=float)
        all_ok = True
        for v in null_b:
            v_np = np.array([float(x) for x in v])
            err = np.linalg.norm(A_np @ v_np)
            if err > 1e-8:
                all_ok = False
        print(f"  Kiểm tra A @ v = 0 với mọi v ∈ N(A): {'PASS' if all_ok else 'FAIL'}")
    else:
        print("   (Không gian nghiệm tầm thường — chỉ có nghiệm x=0)")

    return rank, col_b, row_b, null_b


# TEST 1: Ma trận 3x3 đủ hạng
print_rank_result([[1, 2, 3], [0, 1, 4], [5, 6, 0]], "TEST 1: Ma trận 3x3 đủ hạng (rank=3)")

# TEST 2: Ma trận thiếu hạng (có null space)
print_rank_result([[1, 2, 3], [2, 4, 6], [0, 1, 4]], "TEST 2: Ma trận hạng 2 (có null space)")

# TEST 3: Ma trận không vuông 2x4
print_rank_result([[1, 2, 0, 1], [0, 0, 1, 3]], "TEST 3: Ma trận 2x4 (rank=2, null dim=2)")

# TEST 4: Ma trận toàn số 0
print_rank_result([[0, 0], [0, 0]], "TEST 4: Ma trận 0 (rank=0)")

# TEST 5: Ma trận 4x3
print_rank_result([[1, 0, 2], [0, 1, -1], [2, 1, 3], [1, -1, 4]], "TEST 5: Ma trận 4x3")


  TEST 1: Ma trận 3x3 đủ hạng (rank=3)
  Ma trận A (3x3): [[1, 2, 3], [0, 1, 4], [5, 6, 0]]

  Hạng (tự cài): 3  |  Hạng (NumPy): 3  PASS

  Cơ sở không gian cột C(A) (3 vector):
    ['1', '0', '5']
    ['2', '1', '6']
    ['3', '4', '0']

  Cơ sở không gian hàng R(A) (3 vector):
    ['1', '0', '0']
    ['0', '1', '0']
    ['0', '0', '1']

  Cơ sở không gian nghiệm N(A) (0 vector):
   (Không gian nghiệm tầm thường — chỉ có nghiệm x=0)

  TEST 2: Ma trận hạng 2 (có null space)
  Ma trận A (3x3): [[1, 2, 3], [2, 4, 6], [0, 1, 4]]

  Hạng (tự cài): 2  |  Hạng (NumPy): 2  PASS

  Cơ sở không gian cột C(A) (2 vector):
    ['1', '2', '0']
    ['2', '4', '1']

  Cơ sở không gian hàng R(A) (2 vector):
    ['1', '0', '-5']
    ['0', '1', '4']

  Cơ sở không gian nghiệm N(A) (1 vector):
    ['5', '-4', '1']
  Kiểm tra A @ v = 0 với mọi v ∈ N(A): PASS

  TEST 3: Ma trận 2x4 (rank=2, null dim=2)
  Ma trận A (2x4): [[1, 2, 0, 1], [0, 0, 1, 3]]

  Hạng (tự cài): 2  |  Hạng (NumPy): 2  PASS

  Cơ sở

(3,
 [[1, 0, 2, 1], [0, 1, 1, -1], [2, -1, 3, 4]],
 [[Fraction(1, 1), Fraction(0, 1), Fraction(0, 1)],
  [Fraction(0, 1), Fraction(1, 1), Fraction(0, 1)],
  [Fraction(0, 1), Fraction(0, 1), Fraction(1, 1)]],
 [])

---
## 6. Kiểm Chứng Tổng Hợp (`verify_solution`)

So sánh toàn diện giữa kết quả tự cài đặt và NumPy/SciPy.

In [11]:
import time

print("="*65)
print("  KIỂM CHỨNG TỔNG HỢP — So sánh với NumPy/SciPy")
print("="*65)

test_cases = [
    {
        "label": "Hệ 3x3 cơ bản",
        "A": [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]],
        "b": [8, -11, -3]
    },
    {
        "label": "Hệ 4x4",
        "A": [[4, 3, 2, 1], [3, 4, 3, 2], [2, 3, 4, 3], [1, 2, 3, 4]],
        "b": [10, 12, 12, 10]
    },
    {
        "label": "Hệ 5x5 ngẫu nhiên (seed=0)",
        "A": np.random.RandomState(0).randint(1, 20, (5,5)).tolist(),
        "b": np.random.RandomState(1).randint(1, 20, 5).tolist()
    },
]

results_summary = []
for tc in test_cases:
    print(f"\n {tc['label']}")
    t0 = time.time()
    M, x_str, s = gaussian_eliminate(tc["A"], tc["b"])
    t1 = time.time()
    x_vec = parse_unique_solution(x_str)
    if x_vec:
        err = verify_solution(tc["A"], x_vec, tc["b"])
        results_summary.append((tc["label"], err, (t1-t0)*1000))
    else:
        print(f"  Kết quả: {x_str}")

print(f"\n\n{'='*65}")
print("  Bảng tóm tắt")
print(f"{'='*65}")
print(f"  {'Tên test':<35} {'Sai số':<15} {'Thời gian (ms)'}")
print(f"  {'-'*60}")
for label, err, t in results_summary:
    status = "PASS" if err < 1e-8 else "FAIL"
    print(f"  {label:<35} {err:<15.2e} {t:.3f} ms  {status}")

  KIỂM CHỨNG TỔNG HỢP — So sánh với NumPy/SciPy

 Hệ 3x3 cơ bản
  Nghiệm tự cài đặt  : [2.0, 3.0, -1.0]
  Nghiệm NumPy       : [2.0, 3.0, -1.0]
  Sai số tương đối   : 0.00e+00
  ||x - x_ref||      : 8.38e-16
  Kết quả            : PASS

 Hệ 4x4
  Nghiệm tự cài đặt  : [1.0, 1.0, 1.0, 1.0]
  Nghiệm NumPy       : [1.0, 1.0, 1.0, 1.0]
  Sai số tương đối   : 0.00e+00
  ||x - x_ref||      : 3.85e-16
  Kết quả            : PASS

 Hệ 5x5 ngẫu nhiên (seed=0)
  Nghiệm tự cài đặt  : [-0.724825, 0.488017, 0.099492, 1.061554, 0.817188]
  Nghiệm NumPy       : [-0.724825, 0.488017, 0.099492, 1.061554, 0.817188]
  Sai số tương đối   : 2.57e-11
  ||x - x_ref||      : 3.61e-11
  Kết quả            : PASS


  Bảng tóm tắt
  Tên test                            Sai số          Thời gian (ms)
  ------------------------------------------------------------
  Hệ 3x3 cơ bản                       0.00e+00        0.196 ms  PASS
  Hệ 4x4                              0.00e+00        0.044 ms  PASS
  Hệ 5x5 ngẫu nhi

---
## 7. Kết Luận

### 7.1 Kết quả tổng hợp Phần 1

| Hàm | Trạng thái | Ghi chú |
|---|---|---|
| `gaussian_eliminate` | Hoạt động  | Partial pivoting đúng |
| `back_substitution` | Chính xác |  
| `determinant` | Chính xác | Tính đúng dấu và giá trị |
| `inverse` | Chính xác | Gauss–Jordan với `Fraction` |
| `rank_and_basis` | Chính xác | RREF đúng, các cơ sở hợp lệ |

In [12]:

print("\n  Hoàn thành Part 1 Demo!")
print("  Tất cả các hàm đã được kiểm thử và kiểm chứng với NumPy.\n")



  Hoàn thành Part 1 Demo!
  Tất cả các hàm đã được kiểm thử và kiểm chứng với NumPy.

